# 🛠️ Elastic Detection Rules Converter — Standalone CLI & `.exe` Builder

**Package the TOML ➜ NDJSON converter into one parameterized command-line tool — and optionally a standalone
`.exe` — so you don't have to open a notebook or remember CLI flags every time.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
![Python](https://img.shields.io/badge/python-3.12%2B-blue)
![License](https://img.shields.io/badge/license-MIT-green)

> This notebook builds `elastic_rules_converter.py`, a single script that replaces
> `ElasticDetectionRulesConverter-[Batch Conversion of Multiple TOML Files to NDJSON Files].ipynb` and `ElasticDetectionRulesConverter-[Batch Conversion of Multiple TOML Files to a Single NDJSON File].ipynb` with one
> command you run with flags. It then shows how to package that script into a standalone executable with
> PyInstaller — read the [honest caveats](#honest-caveats) section before you rely on the `.exe` for a fully
> dependency-free workflow.

> ⭐ If this saved you time, please **star** the repo!


## 📑 Table of Contents

1. [Overview](#overview)
2. [Honest Caveats — Read This First](#honest-caveats)
3. [Prerequisites](#prerequisites)
4. [Step 1 — Clone the Repo & Install the CLI](#step-1)
5. [Step 2 — Write the Standalone Script](#step-2)
6. [Step 3 — Use It Like a Normal CLI Tool](#step-3)
7. [Step 4 — Package It as an Executable (PyInstaller)](#step-4)
8. [Step 5 — Download the Executable](#step-5)
9. [Getting a Real Windows `.exe` (CI Build)](#windows-exe)
10. [Troubleshooting](#troubleshooting)
11. [License & Disclaimer](#license)


<a id="overview"></a>
## 📖 Overview

The two companion notebooks in this repo (per-rule export and single-file export) are great for a one-off,
guided run — but if you find yourself running this conversion regularly, opening Colab and clicking through
cells every time gets old fast.

This notebook builds a **standalone command-line script**, `elastic_rules_converter.py`, that does everything
both other notebooks do — single-file *or* per-rule export — behind simple flags:

```bash
python elastic_rules_converter.py --category windows --mode single
python elastic_rules_converter.py --category linux  --mode individual --zip
python elastic_rules_converter.py --category rules  --mode single --output all_rules.ndjson --zip
```

It then shows how to turn that script into a standalone executable with
[PyInstaller](https://pyinstaller.org/), so "lazy researchers" (your words, and honestly, a very reasonable way
to want to work) can run one file with parameters instead of setting up a Python environment by hand.


<a id="honest-caveats"></a>
## ⚠️ Honest Caveats — Read This First

To set expectations correctly:

- **The script (and the `.exe` built from it) still needs Elastic's own tooling present at runtime** — a local
  clone of `elastic/detection-rules` plus `poetry install` already run inside it. This tool doesn't reimplement
  Elastic's TOML parsing/validation logic; it orchestrates Elastic's official CLI, the same way the other two
  notebooks do. That's a deliberate choice: it means results always match what Elastic's own tooling produces,
  even as their rule schema evolves.
- **PyInstaller builds for the operating system it runs on.** If you build the `.exe` inside this Colab notebook
  (which runs on Linux), you'll get a Linux binary, not a native Windows `.exe`, even though the file might be
  named `elastic_rules_converter.exe`. To get a genuine Windows executable, either run the PyInstaller step on a
  Windows machine, or use the GitHub Actions workflow described in [Getting a Real Windows `.exe`](#windows-exe),
  which builds it on an actual Windows runner and publishes it as a download — no build step required on your end.
- In short: this step turns *"open a notebook and run 10 cells"* into *"run one file with flags"* — a real
  usability win — but it is not (yet) a single, fully offline, zero-dependency binary. That would require bundling
  Elastic's entire `detection_rules` package and its dependencies into the executable, which is listed as a
  possible future enhancement.


<a id="prerequisites"></a>
## ✅ Prerequisites

Same as the other two notebooks: Google Colab (recommended) or any Linux/macOS machine with Python 3.12+, `git`,
and internet access.


<a id="step-1"></a>
## 1️⃣ Clone the Repo & Install the CLI

The same one-time setup used by the other two notebooks — the standalone script shells out to this same
installed CLI.


In [ ]:
!pip install -q requests toml poetry
!git clone --depth 1 https://github.com/elastic/detection-rules.git

In [ ]:
%cd detection-rules

In [ ]:
!poetry install

In [ ]:
!poetry run python -m detection_rules --help

<a id="step-2"></a>
## 2️⃣ Write the Standalone Script

Writes `elastic_rules_converter.py` to disk using the `%%writefile` magic. This single file accepts `--category`,
`--mode` (`single` or `individual`), `--output`, and `--zip` — everything both other notebooks do, in one place.


In [ ]:
%%writefile elastic_rules_converter.py
#!/usr/bin/env python3
"""
elastic_rules_converter.py
---------------------------
Standalone, parameterized command-line tool that converts Elastic detection-rules
TOML files into Kibana-importable NDJSON -- either as one combined file per category,
or as one .ndjson per individual rule.

It is a thin wrapper around Elastic's own, officially documented CLI command
(`detection_rules export-rules-from-repo`) -- it does not re-implement rule parsing
or validation itself, so results always match what Elastic's tooling produces.

REQUIREMENTS (on the machine actually running the export):
    - A local clone of https://github.com/elastic/detection-rules  (default: ./detection-rules)
    - Poetry installed, with `poetry install` already run inside that clone

USAGE EXAMPLES
    python elastic_rules_converter.py --category windows --mode single
    python elastic_rules_converter.py --category linux --mode individual --zip
    python elastic_rules_converter.py --category rules --mode single --output all_rules.ndjson --zip
"""

import argparse
import os
import shutil
import subprocess
import sys


def find_toml_files(base_dir):
    """Yield the full path of every .toml file under base_dir."""
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith(".toml"):
                yield os.path.join(root, file)


def run_cli(repo_dir, args):
    """Run `poetry run python -m detection_rules ...` inside repo_dir."""
    cmd = ["poetry", "run", "python", "-m", "detection_rules"] + args
    return subprocess.run(cmd, cwd=repo_dir, capture_output=True, text=True)


def export_single(repo_dir, rules_subdir, output_file):
    """Export an entire category directory into one combined .ndjson file."""
    output_file = os.path.abspath(output_file)
    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)

    print(f"Exporting all rules under '{rules_subdir}' -> {output_file}")
    result = run_cli(repo_dir, ["export-rules-from-repo", "-d", rules_subdir, "-o", output_file])

    if result.returncode != 0:
        print("Export failed:")
        print(result.stderr)
        sys.exit(1)

    print(f"Export succeeded -> {output_file}")
    return output_file


def export_individual(repo_dir, rules_subdir, output_dir):
    """Export every .toml rule in a category directory as its own .ndjson file."""
    output_dir = os.path.abspath(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    rules_full_dir = os.path.join(repo_dir, rules_subdir)
    success, failed = 0, 0

    for rule_path in find_toml_files(rules_full_dir):
        rel_rule_path = os.path.relpath(rule_path, repo_dir)
        output_name = os.path.basename(rule_path).replace(".toml", ".ndjson")
        output_path = os.path.join(output_dir, output_name)

        print(f"Exporting {os.path.basename(rule_path)}...")
        result = run_cli(repo_dir, ["export-rules-from-repo", "-f", rel_rule_path, "-o", output_path])

        if result.returncode == 0:
            success += 1
        else:
            failed += 1
            print(f"  Failed: {rel_rule_path}")
            print(f"  {result.stderr.strip()}")

    print()
    print("=" * 60)
    print(f"Exported : {success}")
    print(f"Failed   : {failed}")
    print("=" * 60)
    return output_dir


def main():
    parser = argparse.ArgumentParser(
        description="Convert Elastic detection-rules TOML files into Kibana-importable NDJSON.",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument(
        "--repo", default="detection-rules",
        help="Path to the cloned elastic/detection-rules repository (default: ./detection-rules)",
    )
    parser.add_argument(
        "--category", default="windows",
        help="Rule category/subfolder under rules/ to export: windows, linux, macos, network, cloud, etc. "
             "Use 'rules' to target every platform at once.",
    )
    parser.add_argument(
        "--mode", choices=["single", "individual"], default="individual",
        help="'single' = one combined .ndjson for the whole category. "
             "'individual' = one .ndjson per rule (default).",
    )
    parser.add_argument(
        "--output", default=None,
        help="Output file (mode=single) or output directory (mode=individual). "
             "Defaults to '<category>_rules.ndjson' or '<category>_NDJSON/'.",
    )
    parser.add_argument(
        "--zip", action="store_true",
        help="Also zip the result when finished.",
    )
    args = parser.parse_args()

    rules_subdir = "rules" if args.category == "rules" else os.path.join("rules", args.category)
    rules_full_dir = os.path.join(args.repo, rules_subdir)

    if not os.path.isdir(rules_full_dir):
        print(f"Could not find '{rules_full_dir}'. Check --repo and --category.")
        sys.exit(1)

    if args.mode == "single":
        output_file = args.output or f"{args.category}_rules.ndjson"
        result_path = export_single(args.repo, rules_subdir, output_file)
        if args.zip:
            base = os.path.splitext(result_path)[0]
            zip_path = shutil.make_archive(base, "zip", os.path.dirname(result_path), os.path.basename(result_path))
            print(f"Zipped -> {zip_path}")
    else:
        output_dir = args.output or f"{args.category}_NDJSON"
        result_dir = export_individual(args.repo, rules_subdir, output_dir)
        if args.zip:
            zip_path = shutil.make_archive(result_dir, "zip", result_dir)
            print(f"Zipped -> {zip_path}")


if __name__ == "__main__":
    main()


<a id="step-3"></a>
## 3️⃣ Use It Like a Normal CLI Tool

Try it out right here in the notebook. This example exports every Windows rule into one combined, zipped
`.ndjson` — the same result as `ElasticDetectionRulesConverter-[Batch Conversion of Multiple TOML Files to a Single NDJSON File].ipynb`, but as a single command.


In [ ]:
!python elastic_rules_converter.py --category windows --mode single --zip

Or the per-rule variant (matches `Elastic_Detection_Rules_Converter.ipynb`):


In [ ]:
!python elastic_rules_converter.py --category windows --mode individual --zip

See every available flag:


In [ ]:
!python elastic_rules_converter.py --help

<a id="step-4"></a>
## 4️⃣ Package It as an Executable (PyInstaller)

[PyInstaller](https://pyinstaller.org/) bundles a Python script and its own Python-level dependencies into one
executable file. Remember: it builds for whatever OS it's *run on* — in Colab, that means a **Linux** binary.


In [ ]:
!pip install -q pyinstaller

In [ ]:
!pyinstaller --onefile --name elastic_rules_converter elastic_rules_converter.py

The resulting binary is written to `dist/elastic_rules_converter`. It still expects to be run
from a location where `--repo` points at a cloned, `poetry install`-ed copy of `detection-rules` (see the
[caveats](#honest-caveats) above) — but now it's a single file you can hand to a colleague, no Python setup
required on their end beyond having `git` + `poetry` available:

```bash
./elastic_rules_converter --category macos --mode single --zip
```


In [ ]:
!ls -lh dist/

<a id="step-5"></a>
## 5️⃣ Download the Executable

Downloads the binary built in Step 4. Again: built here, it's a **Linux** executable — see the next section for
a genuine Windows `.exe`.


In [ ]:
try:
    from google.colab import files
    files.download("dist/elastic_rules_converter")
except ImportError:
    print("Not running in Google Colab — grab the file directly from disk instead:")
    print("  dist/elastic_rules_converter")


<a id="windows-exe"></a>
## 🪟 Getting a Real Windows `.exe` (CI Build)

Since PyInstaller only builds for the OS it runs on, the cleanest way to hand researchers an actual double-click
`elastic_rules_converter.exe` is to build it on a **real Windows machine** — most conveniently, a free GitHub
Actions **`windows-latest`** runner, so nobody has to own a Windows PC to produce it.

A minimal workflow for this repo looks like:

```yaml
# .github/workflows/build-exe.yml
name: Build Windows EXE

on:
  push:
    tags: ["v*"]
  workflow_dispatch:

jobs:
  build:
    runs-on: windows-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Clone detection-rules
        run: git clone --depth 1 https://github.com/elastic/detection-rules.git

      - name: Install Poetry and project dependencies
        run: |
          pip install poetry pyinstaller
          cd detection-rules
          poetry install

      - name: Build the executable
        run: pyinstaller --onefile --name elastic_rules_converter elastic_rules_converter.py

      - name: Upload as a release asset
        uses: softprops/action-gh-release@v2
        with:
          files: dist/elastic_rules_converter.exe
```

Push a version tag (e.g. `v1.0.0`), and this workflow produces a genuine `elastic_rules_converter.exe` on the
repo's **Releases** page automatically — so the "lazy researcher" experience becomes: open Releases, download
the `.exe`, run it with flags. No Python, no Colab, no local build step on their end at all.

> Note: the built `.exe` still needs `git` and `poetry` on the machine it's *run* on, per the caveats above,
> unless a future version bundles the whole `detection_rules` package into the executable itself.


<a id="license"></a>
## ⚖️ License & Disclaimer

This notebook is an independent automation wrapper and is **not officially affiliated with or endorsed by
Elastic**. It automates Elastic's own publicly documented CLI commands from
[`elastic/detection-rules`](https://github.com/elastic/detection-rules), which is separately licensed by Elastic.

Suggested license for this wrapper notebook/repository: MIT.
